# 🔍 Clase 1 — Fundamentos de análisis exploratorio y estadística descriptiva

**Situación:** Eres parte del equipo de analítica de una empresa de servicios logísticos. Antes de diseñar modelos predictivos, necesitas entregar un **primer informe exploratorio** que identifique qué variables son útiles, cómo se comportan los datos y si existen anomalías o patrones críticos.

**Objetivos:**
- Distinguir IDA vs EDA y aplicar ambos
- Análisis univariado: variables numéricas y categóricas
- Análisis multivariado: relaciones entre variables
- Detectar valores atípicos y rutas críticas

> ⚠️ El archivo `entregas_logistica.csv` contiene errores intencionales para practicar diagnóstico.

---
## 📦 Librerías

> Si falta alguna: `pip install pandas matplotlib seaborn`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid', palette='Blues_d')
plt.rcParams['figure.dpi'] = 110

print('✅ Librerías cargadas')
print(f'pandas {pd.__version__} | seaborn {sns.__version__}')

---
## ETAPA 1 — Análisis Inicial de Datos (IDA)
### ¿Están bien cargados los datos?

### 1.1 Carga e inspección general

In [ ]:
df = pd.read_csv('entregas_logistica.csv')

print('=== Primeras filas ===')
print(df.head(10))

In [ ]:
print('=== Estructura del dataset ===')
print(f'Filas: {df.shape[0]} | Columnas: {df.shape[1]}')
print()
print(df.info())

In [ ]:
print('=== Tipos de dato por columna ===')
print(df.dtypes)

### 1.2 Verificar valores faltantes

In [ ]:
print('=== Valores faltantes por columna ===')
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else 'Sin valores faltantes')
print()
print('=== Fila con valor faltante ===')
print(df[df.isnull().any(axis=1)])

### 1.3 Detectar inconsistencias en columnas de texto

In [ ]:
print('=== Valores únicos en region ===')
print(sorted(df['region'].unique()))

print()
print('=== Valores únicos en medio_transporte ===')
print(sorted(df['medio_transporte'].unique()))

### ✏️ Ejercicio IDA — Identifica y corrige:

In [ ]:
# ✏️ Describe los errores que encontraste:
errores = """
Error 1 (valor faltante):          
Error 2 (inconsistencia región):   
Error 3 (inconsistencia transporte):
"""
print(errores)

# Corrección: estandarizar texto
df['region']            = df['region'].str.title()
df['medio_transporte']  = df['medio_transporte'].str.capitalize()

# Imputar tiempo_entrega faltante con la mediana de su región
mediana_region = df.groupby('region')['tiempo_entrega'].transform('median')
df['tiempo_entrega']    = df['tiempo_entrega'].fillna(mediana_region)

print('✅ Dataset limpio — valores únicos región:')
print(sorted(df['region'].unique()))
print('\nValores únicos medio_transporte:')
print(sorted(df['medio_transporte'].unique()))

---
## ETAPA 2 — Análisis Exploratorio de Datos (EDA)
### ¿Qué nos dicen los datos?

### 2.1 Resumen estadístico general

In [ ]:
print('=== Estadística descriptiva — variables numéricas ===')
print(df.describe().round(2))

---
## ETAPA 3 — Análisis Univariado
### Una variable a la vez

### 3.1 Variable numérica continua: `tiempo_entrega`

In [ ]:
print('=== Métricas de tiempo_entrega ===')
t = df['tiempo_entrega']
print(f'Media:              {t.mean():.2f} días')
print(f'Mediana:            {t.median():.2f} días')
print(f'Moda:               {t.mode().iloc[0]:.2f} días')
print(f'Desv. estándar:     {t.std():.2f}')
print(f'Mínimo:             {t.min():.2f} días')
print(f'Máximo:             {t.max():.2f} días')
print(f'Q1 (25%):           {t.quantile(0.25):.2f} días')
print(f'Q3 (75%):           {t.quantile(0.75):.2f} días')
print(f'IQR:                {t.quantile(0.75) - t.quantile(0.25):.2f}')
print()
sesgo = t.mean() - t.median()
print(f'Diferencia media-mediana: {sesgo:.2f} días → {"sesgo positivo (cola derecha)" if sesgo > 0.5 else "distribución aproximadamente simétrica"}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Análisis Univariado — Tiempo de Entrega (días)', fontsize=12, fontweight='bold')

# Histograma + KDE
sns.histplot(df['tiempo_entrega'], bins=20, kde=True, ax=axes[0], color='#2E75B6')
axes[0].axvline(t.mean(),   color='red',    linestyle='--', label=f'Media {t.mean():.1f}')
axes[0].axvline(t.median(), color='orange', linestyle='--', label=f'Mediana {t.median():.1f}')
axes[0].set_title('Distribución (histograma + KDE)')
axes[0].set_xlabel('Días')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# Boxplot
sns.boxplot(y=df['tiempo_entrega'], ax=axes[1], color='#BDD7EE')
axes[1].set_title('Boxplot — Detección de outliers')
axes[1].set_ylabel('Días')

plt.tight_layout()
plt.show()

### ✏️ Ejercicio 3.1 — Responde:

In [ ]:
# ✏️ ¿La distribución es simétrica o sesgada? ¿Qué indica eso para el negocio?
r_dist = ""

# ✏️ ¿Hay valores extremos visibles en el boxplot? ¿Qué representan?
r_outliers = ""

# ✏️ ¿Por qué es útil comparar media vs mediana en este contexto?
r_media_med = ""

print('Distribución:', r_dist)
print('Outliers:    ', r_outliers)
print('Media vs Med:', r_media_med)

### 3.2 Variable numérica: `distancia_km`

In [ ]:
print('=== Métricas de distancia_km ===')
d = df['distancia_km']
print(d.describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df['distancia_km'], bins=20, kde=True, ax=axes[0], color='#70AD47')
axes[0].set_title('Distribución de distancia (km)')
axes[0].set_xlabel('km')

sns.boxplot(y=df['distancia_km'], ax=axes[1], color='#E2EFDA')
axes[1].set_title('Boxplot — distancia_km')
plt.tight_layout()
plt.show()

### 3.3 Variable categórica: `region`

In [ ]:
print('=== Frecuencia por región ===')
freq_region = df['region'].value_counts()
freq_pct    = df['region'].value_counts(normalize=True).mul(100).round(1)
tabla_region = pd.DataFrame({'Frecuencia': freq_region, 'Porcentaje (%)': freq_pct})
print(tabla_region)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Análisis Univariado — Región', fontsize=12, fontweight='bold')

sns.countplot(data=df, y='region', order=freq_region.index, ax=axes[0],
              palette='Blues_r')
axes[0].set_title('Entregas por región')
axes[0].set_xlabel('Cantidad')

axes[1].pie(freq_region.values, labels=freq_region.index,
            autopct='%1.1f%%', startangle=90,
            colors=sns.color_palette('Blues_r', len(freq_region)))
axes[1].set_title('Distribución porcentual')

plt.tight_layout()
plt.show()

### 3.4 Variable categórica: `medio_transporte` y `estado`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Análisis Univariado — Categóricas', fontsize=12, fontweight='bold')

order_medio = df['medio_transporte'].value_counts().index
sns.countplot(data=df, x='medio_transporte', order=order_medio,
              ax=axes[0], palette='Set2')
axes[0].set_title('Medio de transporte')
axes[0].set_xlabel('')

order_estado = df['estado'].value_counts().index
sns.countplot(data=df, x='estado', order=order_estado,
              ax=axes[1], palette='coolwarm')
axes[1].set_title('Estado de la entrega')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

### ✏️ Ejercicio 3.4 — Responde:

In [ ]:
# ✏️ ¿Qué región concentra más entregas? ¿Es esperable?
r_region = ""

# ✏️ ¿Hay desbalance notable en los medios de transporte?
r_medio = ""

# ✏️ ¿Qué porcentaje de entregas tiene retraso? ¿Es preocupante?
pct_retraso = (df['estado'] == 'Con retraso').sum() / len(df) * 100
print(f'% de entregas con retraso: {pct_retraso:.1f}%')

print('Región:', r_region)
print('Medio: ', r_medio)

### 3.5 Detección de outliers con regla IQR

In [ ]:
Q1  = df['tiempo_entrega'].quantile(0.25)
Q3  = df['tiempo_entrega'].quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR
limite_inferior = Q1 - 1.5 * IQR

print(f'Q1: {Q1:.2f} | Q3: {Q3:.2f} | IQR: {IQR:.2f}')
print(f'Límite inferior: {limite_inferior:.2f} días')
print(f'Límite superior: {limite_superior:.2f} días')
print()

outliers = df[df['tiempo_entrega'] > limite_superior]
print(f'Entregas con demoras atípicas (outliers): {len(outliers)}')
print(outliers[['id_entrega','region','medio_transporte','distancia_km','tiempo_entrega']].to_string(index=False))

In [ ]:
# Entregas con tiempo > 7 días (umbral operacional)
print('=== Entregas con tiempo_entrega > 7 días ===')
atrasos = df[df['tiempo_entrega'] > 7]
print(f'Total: {len(atrasos)} entregas')
print()
print(atrasos[['region', 'tiempo_entrega']].value_counts().head(10))

---
## ETAPA 4 — Análisis Multivariado
### ¿Cómo se relacionan las variables entre sí?

### 4.1 Relación entre distancia y tiempo de entrega (scatter)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Análisis Multivariado — Distancia vs Tiempo de Entrega', fontsize=12, fontweight='bold')

# Scatter coloreado por región
paleta = sns.color_palette('tab10', n_colors=df['region'].nunique())
sns.scatterplot(data=df, x='distancia_km', y='tiempo_entrega',
                hue='region', palette=paleta, alpha=0.75, s=70, ax=axes[0])
axes[0].set_title('Por región')
axes[0].set_xlabel('Distancia (km)')
axes[0].set_ylabel('Tiempo entrega (días)')
axes[0].legend(fontsize=7, title='Región')

# Scatter con línea de tendencia por medio de transporte
sns.lmplot(data=df, x='distancia_km', y='tiempo_entrega',
           hue='medio_transporte', scatter_kws={'alpha': 0.5},
           height=4.5, aspect=1.3)
plt.title('Tendencia por medio de transporte')
plt.xlabel('Distancia (km)')
plt.ylabel('Tiempo entrega (días)')
plt.tight_layout()
plt.show()

### 4.2 Boxplot por región: comparar distribuciones

In [ ]:
orden_region = df.groupby('region')['tiempo_entrega'].median().sort_values().index

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Tiempo de Entrega por Región y Medio de Transporte', fontsize=12, fontweight='bold')

sns.boxplot(data=df, x='region', y='tiempo_entrega',
            order=orden_region, palette='Blues', ax=axes[0])
axes[0].set_title('Por región (ordenado por mediana)')
axes[0].set_xlabel('Región')
axes[0].set_ylabel('Días')
axes[0].tick_params(axis='x', rotation=30)

sns.boxplot(data=df, x='medio_transporte', y='tiempo_entrega',
            palette='Set2', ax=axes[1])
axes[1].set_title('Por medio de transporte')
axes[1].set_xlabel('')
axes[1].set_ylabel('Días')

plt.tight_layout()
plt.show()

### 4.3 Matriz de correlación (heatmap)

In [ ]:
numericas = df[['distancia_km', 'tiempo_entrega', 'valor_pedido']]
corr = numericas.corr()

plt.figure(figsize=(6, 4))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues',
            vmin=-1, vmax=1, linewidths=0.5)
plt.title('Matriz de correlación', fontweight='bold')
plt.tight_layout()
plt.show()

print('=== Correlación numérica ===')
print(corr.round(3))

### 4.4 Promedio de tiempo por región y medio de transporte

In [ ]:
tabla_cruzada = df.groupby(['region', 'medio_transporte'])['tiempo_entrega'] \
                  .mean().round(1).unstack()

print('=== Tiempo promedio por región y medio de transporte ===')
print(tabla_cruzada)

tabla_cruzada.plot(kind='bar', figsize=(12, 5), colormap='Set2', edgecolor='white')
plt.title('Tiempo promedio de entrega por región y medio de transporte', fontweight='bold')
plt.xlabel('Región')
plt.ylabel('Días promedio')
plt.xticks(rotation=30)
plt.legend(title='Medio')
plt.tight_layout()
plt.show()

### ✏️ Ejercicio 4 — Responde:

In [ ]:
# ✏️ 1. ¿Existe una relación lineal entre distancia y tiempo de entrega?
r1 = ""

# ✏️ 2. ¿Qué región presenta los tiempos de entrega más altos? ¿A qué podría deberse?
r2 = ""

# ✏️ 3. ¿El medio de transporte influye en el tiempo de entrega? ¿Por qué?
r3 = ""

# ✏️ 4. ¿La correlación implica causalidad? Razona tu respuesta.
r4 = ""

print('--- REFLEXIONES MULTIVARIADAS ---')
for i, r in enumerate([r1, r2, r3, r4], 1):
    print(f'{i}. {r}')

---
## ETAPA 5 — Informe preliminar ejecutivo

In [ ]:
print('=' * 60)
print('     INFORME EXPLORATORIO — OPERACIONES LOGÍSTICAS')
print('=' * 60)
print(f'Registros analizados:           {len(df)}')
print(f'Período:                        Mayo 2024')
print()
print('TIEMPO DE ENTREGA')
print(f'  Promedio:                     {df["tiempo_entrega"].mean():.1f} días')
print(f'  Mediana:                      {df["tiempo_entrega"].median():.1f} días')
print(f'  Máximo (con outlier):         {df["tiempo_entrega"].max():.1f} días')
print(f'  Entregas atípicas (IQR):      {len(outliers)}')
print(f'  Entregas > 7 días:            {len(atrasos)}')
print()
print('REGIONES CRÍTICAS (mayor tiempo promedio)')
top3 = df.groupby('region')['tiempo_entrega'].mean().nlargest(3)
for reg, val in top3.items():
    print(f'  {reg:<20} {val:.1f} días promedio')
print()
print('CORRELACIÓN distancia ↔ tiempo:')
corr_val = df['distancia_km'].corr(df['tiempo_entrega'])
print(f'  Pearson r = {corr_val:.3f} → {"relación fuerte" if abs(corr_val) > 0.7 else "relación moderada" if abs(corr_val) > 0.4 else "relación débil"}')
print('=' * 60)

### ✏️ Preguntas de cierre:

In [ ]:
# ✏️ 1. ¿Cuál es el comportamiento general del tiempo de entrega?
c1 = ""

# ✏️ 2. ¿Existen rutas críticas con valores atípicos? ¿Cuáles?
c2 = ""

# ✏️ 3. ¿Qué variables podrían estar influyendo más en las demoras?
c3 = ""

# ✏️ 4. ¿Qué diferencia hay entre IDA y EDA? ¿Cuándo aplicarías cada uno?
c4 = ""

# ✏️ 5. ¿Por qué el EDA es un paso crítico antes del modelado?
c5 = ""

print('--- CONCLUSIONES DEL INFORME ---')
for i, c in enumerate([c1, c2, c3, c4, c5], 1):
    print(f'{i}. {c}')

---
## 📋 Resumen de técnicas y herramientas

| Etapa | Técnica | Función | Código |
|-------|---------|---------|--------|
| **IDA** | Estructura | Verificar carga | `df.shape`, `df.dtypes`, `df.info()` |
| **IDA** | Nulos | Detectar faltantes | `df.isnull().sum()` |
| **IDA** | Inconsistencias | Estandarizar texto | `str.title()`, `str.capitalize()` |
| **EDA Univariado** | Estadísticas | Describir variable | `df.describe()`, `.mean()`, `.std()` |
| **EDA Univariado** | Distribución | Visualizar forma | `sns.histplot()`, `sns.boxplot()` |
| **EDA Univariado** | Categórica | Frecuencias | `value_counts()`, `sns.countplot()` |
| **EDA Univariado** | Outliers | Detectar extremos | Regla IQR: `Q3 + 1.5*IQR` |
| **EDA Multivariado** | Correlación | Medir relación | `.corr()`, `sns.heatmap()` |
| **EDA Multivariado** | Scatter | Ver asociación | `sns.scatterplot()`, `sns.lmplot()` |
| **EDA Multivariado** | Comparación | Grupos | `groupby()`, `sns.boxplot()` |

> 💡 **IDA vs EDA:** IDA pregunta *¿están bien los datos?* — EDA pregunta *¿qué dicen los datos?*

> 💡 **Correlación ≠ causalidad:** Una correlación alta entre dos variables no implica que una cause la otra. Siempre busca explicaciones alternativas.